In [ ]:
import numpy as np
import pandas as pd
import os
import json

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


# ============================================================
# Loaders
# ============================================================

def load_and_merge_radiomics(paths_dict):
    dfs = []
    for modality, path in paths_dict.items():
        df = pd.read_csv(path)
        drop_cols = [c for c in ["sex", "age", "modality"] if c in df.columns]
        df = df.drop(columns=drop_cols, errors="ignore").copy()
        rad_cols = [c for c in df.columns if c != "case_id"]
        df = df.rename(columns={c: f"{modality}_{c}" for c in rad_cols})
        dfs.append(df)

    final_df = dfs[0]
    for d in dfs[1:]:
        final_df = pd.merge(final_df, d, on="case_id", how="inner")
    return final_df


def load_image_features(case_id, base_dir="image_features"):
    modalities = ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]
    out = {}
    for m in modalities:
        p = os.path.join(base_dir, str(case_id), m, "image.npy")
        try:
            feats = np.load(p)
            if feats.ndim > 1:
                feats = feats.flatten()
            out[m] = feats
        except FileNotFoundError:
            out[m] = None
    return out


def extract_all_image_features(df_cases, base_dir="image_features"):
    all_features, case_ids = [], []
    ref_dim = None

    for case_id in df_cases["case_id"].values:
        img = load_image_features(case_id, base_dir)
        if all(v is not None for v in img.values()):
            combined = np.concatenate([img[m] for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]])
            if ref_dim is None:
                ref_dim = combined.shape[0]
            all_features.append(combined)
        else:
            all_features.append(None)
        case_ids.append(case_id)

    if ref_dim is None:
        return None

    filled = [f if f is not None else np.full(ref_dim, np.nan) for f in all_features]
    cols = [f"img_feat_{i}" for i in range(ref_dim)]
    out = pd.DataFrame(filled, columns=cols)
    out.insert(0, "case_id", case_ids)
    return out


def load_reports(json_path):
    """Load raw radiology findings. Extract only 'finding' from dict entries; 
    never include 'impression' (contains diagnostic leakage)."""
    with open(json_path, "r") as f:
        data = json.load(f)
    out = {}
    for case_id, entry in data.items():
        if not isinstance(entry, dict):
            out[str(case_id)] = ""
            continue
        r = entry.get("report", "")
        if isinstance(r, str):
            out[str(case_id)] = r
        elif isinstance(r, dict):
            # Take ONLY the findings section; deliberately exclude 'impression'
            finding = r.get("finding") or r.get("findings") or ""
            out[str(case_id)] = finding if isinstance(finding, str) else ""
        else:
            out[str(case_id)] = ""
    return out

LOCATION_KEYWORDS = [
    "frontal", "parietal", "temporal", "occipital", "insular", "insula",
    "falx", "cerebri", "cranial fossa", "convexity", "parasagittal",
    "cerebellopontine", "tentorial", "tentorium", "sphenoid",
    "petrous", "clinoid", "meningeal", "dural", "sagittal sinus",
    "sellar", "sella", "suprasellar", "parasellar", "intrasellar",
    "turcica", "pituitary",
    "pineal", "ventricle", "ventricular", "aqueduct", "choroid",
    "third ventricle", "fourth ventricle", "lateral ventricle",
    "basal ganglia", "corpus callosum", "subcortical", "thalamus",
    "hippocamp", "cingulate",
    "cerebellar", "cerebellum", "brainstem", "pons", "pontine",
    "medulla", "midbrain", "clivus",
    "left", "right", "bilateral", "midline", "multiple",
]


def prepare_clinical_features(csv_path, age_median=None):
    clin = pd.read_csv(csv_path).copy()

    if age_median is None:
        age_median = clin["Age"].median()
    clin["Age"] = clin["Age"].fillna(age_median)

    loc = clin["Tumor Location"].fillna("").str.lower()
    for kw in LOCATION_KEYWORDS:
        clin[f"loc_{kw}"] = loc.str.contains(kw, regex=False).astype(int)
    clin["loc_missing"] = clin["Tumor Location"].isna().astype(int)
    clin["loc_num_regions"] = loc.str.count(",") + (loc.str.len() > 0).astype(int)
    clin = clin.drop(columns=["Tumor Location"])

    cat_cols = ["Sex",
                "Signal Intensity (T1)", "Signal Intensity (T1c)",
                "Signal Intensity (T2)", "Signal Intensity (T2-FLAIR)"]
    cat_cols = [c for c in cat_cols if c in clin.columns]
    clin = pd.get_dummies(clin, columns=cat_cols, dummy_na=True)

    bool_cols = clin.select_dtypes(include=["bool"]).columns
    clin[bool_cols] = clin[bool_cols].astype(int)

    return clin, age_median


def align_clinical(df_, cols):
    for c in cols:
        if c not in df_.columns:
            df_[c] = 0
    return df_[["case_id"] + cols]


# ============================================================
# Labels (from JSON — same files that hold the reports)
# ============================================================

train_json = pd.read_json("train.json")
val_json = pd.read_json("val.json")
train_class = pd.DataFrame(train_json.T.Overall_class, columns=["Overall_class"])
valid_class = pd.DataFrame(val_json.T.Overall_class, columns=["Overall_class"])

le = LabelEncoder()
y_train = le.fit_transform(train_class["Overall_class"])
y_valid = le.transform(valid_class["Overall_class"])
n_classes = len(le.classes_)

print("Class label -> encoded index:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")

print("\nTraining class counts:")
class_counts = np.bincount(y_train)
for cls, cnt in zip(le.classes_, class_counts):
    print(f"  {cls}: {cnt}")

min_class = class_counts.min()
smote_k = max(1, min(5, min_class - 1))
print(f"\nUsing SMOTE k_neighbors={smote_k} (smallest class has {min_class} samples)")

# ============================================================
# Load reports from the JSON files
# ============================================================
print("\nLoading raw reports from JSON...")
reports_train_dict = load_reports("train.json")
reports_val_dict   = load_reports("val.json")
reports_test_dict  = load_reports("test.json")
print(f"Reports loaded -> train: {len(reports_train_dict)}, "
      f"val: {len(reports_val_dict)}, test: {len(reports_test_dict)}")
# Sanity check: show one report
example_key = list(reports_train_dict.keys())[0]
print(f"Example report (case_id={example_key}):")
print(f"  {reports_train_dict[example_key][:250]}...")

# ============================================================
# Load tabular data
# ============================================================

train_paths = {m: f"{m}_radiomics_train.csv" for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]}
val_paths   = {m: f"{m}_radiomics_val.csv"   for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]}
test_paths  = {m: f"{m}_radiomics_test.csv"  for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]}

rad_train = load_and_merge_radiomics(train_paths)
rad_val   = load_and_merge_radiomics(val_paths)
rad_test  = load_and_merge_radiomics(test_paths)

clin_train, age_med = prepare_clinical_features("train_patient_info.csv")
clin_val,   _       = prepare_clinical_features("val_patient_info.csv",  age_median=age_med)
clin_test,  _       = prepare_clinical_features("test_patient_info.csv", age_median=age_med)

train_clin_cols = [c for c in clin_train.columns if c != "case_id"]
clin_val  = align_clinical(clin_val,  train_clin_cols)
clin_test = align_clinical(clin_test, train_clin_cols)

print(f"\nClinical feature count: {len(train_clin_cols)}")

print("\nExtracting image features...")
img_train = extract_all_image_features(clin_train)
img_val   = extract_all_image_features(clin_val)
img_test  = extract_all_image_features(clin_test)

# ============================================================
# Merge + identify blocks
# ============================================================

def merge_all(clin_df, rad_df, img_df):
    df = pd.merge(clin_df, rad_df, on="case_id", how="inner")
    if img_df is not None:
        df = pd.merge(df, img_df, on="case_id", how="left")
    return df

df_train = merge_all(clin_train, rad_train, img_train)
df_val   = merge_all(clin_val,   rad_val,   img_val)
df_test  = merge_all(clin_test,  rad_test,  img_test)

clinical_cols  = list(train_clin_cols)
img_cols       = [c for c in df_train.columns if c.startswith("img_feat_")]
radiomics_cols = [c for c in df_train.columns
                  if c not in clinical_cols + img_cols + ["case_id"]]


def align(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df[["case_id"] + cols]

all_cols = clinical_cols + radiomics_cols + img_cols
df_val  = align(df_val,  all_cols)
df_test = align(df_test, all_cols)

print(f"Blocks -> clinical: {len(clinical_cols)}, radiomics: {len(radiomics_cols)}, image: {len(img_cols)}")

# ============================================================
# Align reports to dataframe row order
# ============================================================
train_reports = [reports_train_dict.get(str(cid), "") for cid in df_train["case_id"].values]
val_reports   = [reports_val_dict.get(str(cid),   "") for cid in df_val["case_id"].values]
test_reports  = [reports_test_dict.get(str(cid),  "") for cid in df_test["case_id"].values]

print(f"\nReports aligned -> train: {len(train_reports)}, val: {len(val_reports)}, test: {len(test_reports)}")
print(f"Empty reports -> train: {sum(1 for r in train_reports if not r)}, "
      f"val: {sum(1 for r in val_reports if not r)}, "
      f"test: {sum(1 for r in test_reports if not r)}")

# ============================================================
# TF-IDF on reports
# ============================================================
print("\nFitting TF-IDF on reports...")
tfidf = TfidfVectorizer(
    max_features=1500,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    lowercase=True,
    stop_words="english",
    sublinear_tf=True,
)
Xt_tr = tfidf.fit_transform(train_reports).toarray().astype(np.float32)
Xt_va = tfidf.transform(val_reports).toarray().astype(np.float32)
Xt_te = tfidf.transform(test_reports).toarray().astype(np.float32)

print(f"TF-IDF shape: train {Xt_tr.shape}, val {Xt_va.shape}, test {Xt_te.shape}")
print(f"Example features: {tfidf.get_feature_names_out()[:20].tolist()}")

# ============================================================
# Per-block preprocessing (clinical / radiomics / image)
# ============================================================

def impute_block(train_block, *others):
    imp = SimpleImputer(strategy="median")
    tr = imp.fit_transform(train_block)
    ot = [imp.transform(b) for b in others]
    return tr, ot

# Clinical
Xc_tr, (Xc_va, Xc_te) = impute_block(
    df_train[clinical_cols].values,
    df_val[clinical_cols].values,
    df_test[clinical_cols].values,
)

# Radiomics
Xr_tr_raw, (Xr_va_raw, Xr_te_raw) = impute_block(
    df_train[radiomics_cols].values,
    df_val[radiomics_cols].values,
    df_test[radiomics_cols].values,
)
rad_scaler = StandardScaler()
Xr_tr_s = rad_scaler.fit_transform(Xr_tr_raw)
Xr_va_s = rad_scaler.transform(Xr_va_raw)
Xr_te_s = rad_scaler.transform(Xr_te_raw)

K_RAD_MAX = min(300, Xr_tr_s.shape[1])
selector = SelectKBest(score_func=f_classif, k=K_RAD_MAX)
Xr_tr_sel = selector.fit_transform(Xr_tr_s, y_train)
Xr_va_sel = selector.transform(Xr_va_s)
Xr_te_sel = selector.transform(Xr_te_s)
rad_rank = np.argsort(selector.scores_[selector.get_support()])[::-1]

# Image
img_missing_train = df_train[img_cols].isna().all(axis=1).astype(int).values.reshape(-1, 1)
img_missing_val   = df_val[img_cols].isna().all(axis=1).astype(int).values.reshape(-1, 1)
img_missing_test  = df_test[img_cols].isna().all(axis=1).astype(int).values.reshape(-1, 1)

Xi_tr_raw, (Xi_va_raw, Xi_te_raw) = impute_block(
    df_train[img_cols].values,
    df_val[img_cols].values,
    df_test[img_cols].values,
)
img_scaler = StandardScaler()
Xi_tr_s = img_scaler.fit_transform(Xi_tr_raw)
Xi_va_s = img_scaler.transform(Xi_va_raw)
Xi_te_s = img_scaler.transform(Xi_te_raw)

N_COMP_MAX = min(256, Xi_tr_s.shape[1], Xi_tr_s.shape[0] - 1)
pca_img = PCA(n_components=N_COMP_MAX, random_state=42)
Xi_tr_pca = pca_img.fit_transform(Xi_tr_s)
Xi_va_pca = pca_img.transform(Xi_va_s)
Xi_te_pca = pca_img.transform(Xi_te_s)
print(f"Image PCA max dims: {N_COMP_MAX} (cumulative EV: {pca_img.explained_variance_ratio_.sum():.2%})")


def assemble(k_rad, n_img):
    """Combine clinical + radiomics (top-k) + image PCA (first n) + TF-IDF + missing flag."""
    rad_idx = rad_rank[:k_rad]
    X_tr = np.hstack([Xc_tr, Xr_tr_sel[:, rad_idx], Xi_tr_pca[:, :n_img], Xt_tr, img_missing_train])
    X_va = np.hstack([Xc_va, Xr_va_sel[:, rad_idx], Xi_va_pca[:, :n_img], Xt_va, img_missing_val])
    X_te = np.hstack([Xc_te, Xr_te_sel[:, rad_idx], Xi_te_pca[:, :n_img], Xt_te, img_missing_test])
    return X_tr, X_va, X_te


# ============================================================
# Randomized search with SMOTE in the CV pipeline
# ============================================================

k_rad_grid = sorted(set(min(K_RAD_MAX, k) for k in [100, 200]))
n_img_grid = [64, 128]

xgb_param_dist = {
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__learning_rate": [0.03, 0.05, 0.07, 0.1],
    "xgb__n_estimators": [400, 600, 900],
    "xgb__subsample": [0.7, 0.8, 1.0],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb__min_child_weight": [1, 3, 5],
    "xgb__gamma": [0, 0.3, 0.5],
    "xgb__reg_alpha": [0, 0.001, 0.01, 0.1],
    "xgb__reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_score = -np.inf
best_cfg = None

for k_rad in k_rad_grid:
    for n_img in n_img_grid:
        n_img_eff = min(n_img, N_COMP_MAX)
        X_tr, _, _ = assemble(k_rad, n_img_eff)

        pipe = ImbPipeline([
            ("smote", SMOTE(random_state=42, k_neighbors=smote_k)),
            ("xgb", XGBClassifier(
                objective="multi:softprob",
                num_class=n_classes,
                random_state=42,
                eval_metric="mlogloss",
                tree_method="hist",
                device="cuda",
                n_jobs=1,
            )),
        ])

        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=xgb_param_dist,
            n_iter=15,
            scoring="f1_macro",
            cv=cv,
            n_jobs=1,
            verbose=0,
            random_state=42,
        )
        search.fit(X_tr, y_train)
        print(f"k_rad={k_rad:>3}, n_img={n_img_eff:>3}  ->  CV f1_macro = {search.best_score_:.4f}")

        if search.best_score_ > best_score:
            best_score = search.best_score_
            best_cfg = (k_rad, n_img_eff, search.best_params_)

k_rad_best, n_img_best, best_params = best_cfg
print(f"\nBest config: k_rad={k_rad_best}, n_img={n_img_best}")
print(f"Best CV f1_macro: {best_score:.4f}")
print(f"Best XGB params: {best_params}")

# ============================================================
# Final fit
# ============================================================
X_train_final, X_val_final, X_test_final = assemble(k_rad_best, n_img_best)
print(f"\nFinal matrix: train {X_train_final.shape}, val {X_val_final.shape}, test {X_test_final.shape}")

smote_final = SMOTE(random_state=42, k_neighbors=smote_k)
X_train_res, y_train_res = smote_final.fit_resample(X_train_final, y_train)
print(f"After SMOTE: {X_train_res.shape}, class counts: {np.bincount(y_train_res).tolist()}")

xgb_params = {k.replace("xgb__", ""): v for k, v in best_params.items()}

final_model = XGBClassifier(
    objective="multi:softprob",
    num_class=n_classes,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    early_stopping_rounds=30,
    **xgb_params,
)
final_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_val_final, y_valid)],
    verbose=False,
)

# ============================================================
# Evaluate
# ============================================================
y_val_pred = final_model.predict(X_val_final)
print(f"\nValidation f1_macro:    {f1_score(y_valid, y_val_pred, average='macro'):.4f}")
print(f"Validation f1_weighted: {f1_score(y_valid, y_val_pred, average='weighted'):.4f}")
print("\nClassification Report (Validation):")
print(classification_report(y_valid, y_val_pred, target_names=le.classes_))

print("Confusion Matrix (rows=true, cols=predicted):")
cm = confusion_matrix(y_valid, y_val_pred)
print(pd.DataFrame(cm, index=le.classes_, columns=le.classes_))

y_test_pred = final_model.predict(X_test_final)
y_test_labels = le.inverse_transform(y_test_pred)

submission = pd.DataFrame({
    "case_id": df_test["case_id"].values,
    "Overall_class": y_test_labels,
})
submission.to_csv("submission_with_reports.csv", index=False)
print("\nSaved submission_with_reports.csv")

Class label -> encoded index:
  0: Brain Metastase Tumour
  1: Glioma
  2: Meningioma
  3: Pineal tumour and Choroid plexus tumour
  4: Tumors of the sellar region

Training class counts:
  Brain Metastase Tumour: 252
  Glioma: 924
  Meningioma: 728
  Pineal tumour and Choroid plexus tumour: 23
  Tumors of the sellar region: 56

Using SMOTE k_neighbors=5 (smallest class has 23 samples)

Loading raw reports from JSON...
Reports loaded -> train: 1983, val: 283, test: 378
Example report (case_id=7):
  In Right frontal region, there is a mass lesions with hypointense-isointense in T1, hyperintense-isointense in T2, heterogenous singal in FLAIR. After contrast administration, there is heterogeneous significant enhancement. Midline structures shift t...

Clinical feature count: 86

Extracting image features...
Blocks -> clinical: 86, radiomics: 20, image: 8192

Reports aligned -> train: 1983, val: 283, test: 378
Empty reports -> train: 0, val: 0, test: 0

Fitting TF-IDF on reports...
TF-IDF 

/home/ubuntu/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [06:44:55] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [5]:
import json
from collections import Counter

for path in ["train.json", "val.json", "test.json"]:
    with open(path) as f:
        data = json.load(f)
    types = Counter(type(entry.get("report")).__name__ for entry in data.values())
    print(f"{path}: {dict(types)}")
    # If any dicts, check their structure
    for cid, entry in data.items():
        r = entry.get("report")
        if isinstance(r, dict):
            print(f"  {path} case_id={cid} keys: {list(r.keys())}")
            break

train.json: {'str': 1983}
val.json: {'str': 283}
test.json: {'str': 367, 'dict': 11}
  test.json case_id=14524495 keys: ['finding', 'impression']


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
cm = confusion_matrix(y_valid, y_val_pred)
print(pd.DataFrame(cm, index=le.classes_, columns=le.classes_))

In [2]:
import os
os.chdir("/home/ubuntu")

In [2]:
import xgboost
print(xgboost.__version__)

3.2.0
